# Intro

## Strategy requirements:
- Maximal profit + minimal loss: Entries only on strong confirmation. Exits use reverse signal OR ATR-based trailing stop (never lets winners turn into losers).
- Consequent logic: Stateful position manager → you can only enter when flat. Every entry must be followed by an exit before the next entry. No two entries in a row.
- Long & Short: All algorithms trade both directions.
- Two modes:
    - Historical (known data): Full look-back on complete candle history → perfect for backtesting/visualization.
    - Live (unknown data): Incremental processing (polls Bybit every interval, decides only on the latest candle → no lookahead).
- Visualization: Interactive Plotly chart with candles + numbered markers (Long Entry 1, Short Exit 1, Long Entry 2, …). Works for both modes.

## Best Algorithms for BTCUSDT Perpetual Entry/Exit Points

Three current 2025-2026 professional strategies for BTCUSDT perpetuals on Bybit (breakout, momentum, trend-following):
1. Swing Level Breakout (Price Action) \
Best for scalping (1m-5m) and intraday (15m-1h). Uses dynamic swing highs/lows (like the level detector).
2. EMA Crossover + RSI Filter \
Momentum strategy, excellent for scalping and intraday across all TFs. \
Filters false signals with RSI.
3. SuperTrend Trend-Following \
Pure trend strategy with built-in ATR trailing stop. \
Extremely popular in crypto perpetuals for all timeframes; maximizes profit by riding trends while cutting losses fast.

## How Each Algorithm Determines Entry/Exit

All strategies use ATR(14) for dynamic stops/tolerance (adapts to volatility).

1. Swing Level Breakout (builds directly on level detector code)

- Detects swing highs (resistance) and lows (support) with configurable left/right window.
- Long Entry: Price closes above a significant resistance level + confirmation candle.
- Short Entry: Price closes below a significant support level + confirmation candle.
- Exit Long: Price closes below next support or ATR trailing stop hit.
- Exit Short: Price closes above next resistance or ATR trailing stop hit.
- Ideal for scalping/intraday when price respects liquidity levels.

2. EMA Crossover + RSI Filter

- Fast EMA (9) / Slow EMA (21) – standard for crypto.
- Long Entry: Fast EMA crosses above Slow EMA AND RSI(14) < 70 (not overbought).
- Short Entry: Fast EMA crosses below Slow EMA AND RSI(14) > 30 (not oversold).
- Exit: Reverse crossover OR price hits ATR-based trailing stop.
- Excellent filter reduces whipsaws in ranging markets.

3. SuperTrend (most popular in 2025-2026 perpetuals)

- Combines ATR volatility bands with trend direction.
- Long Entry: SuperTrend flips from red to green (price closes above upper band).
- Short Entry: SuperTrend flips from green to red (price closes below lower band).
- Exit: SuperTrend flips opposite OR the built-in ATR trailing stop is hit (the SuperTrend line itself acts as dynamic stop).
- Extremely clean, low-lag, and maximizes trend capture while protecting capital.

## Strategy Variants

__Separate mode classes:__
- one signal detector
- one entry signal
- separate mode classes for different directions
- in '_inv' mode the same signal is traded in the opposite direction
- -> 1 signal --> several directions
- -> a direction flip, not different entry logic

3 entry modes for one strategy class:
1. regular
2. inverse
3. adaptive

__Separate strategy classes:__
- one signal detector
- different entry signals on the same detector
- -> several different signals --> several different strategies
- ->  different entries

## Strategy Configuration

Configuration options:
1. __Centralized__ \
Notebooks read the global ACTIVE (data), StrategyConfig() (signal knobs), ACTIVE_TRADE (trade params), and each strategy's assigned exit preset. \
To change anything: edit a *_configurator.py file, which mutates the project-wide default for every notebook at once. \
There is no way to tune one notebook in isolation.

*Example for a new exit preset:* \
In strategy_configurator.py: add the entry in PER_STRATEGY_EXIT. You can assign any preset name there. \
PER_STRATEGY_EXIT = { \
    "order_block": "atr_stop_rr2",         # was "structural" — now uses a different preset by default \
    ... \
}

2. __Per notebook__ \
Each strategy notebook is a self-contained tuning surface for data, strategy, exits, and trade parameters, \
where local edits override the centralized defaults (and any default baked into BaseStrategy/exit_policy_for) \
without touching the configurators — while leaving the centralized path fully intact when the local edits are left blank.

__Each notebook's Configuration module__ is split into two:
- Configuration: automatic
    - This is the centralized default.
    - Defines the canonical handles from the three configurators.
- Configuration: manual
    - Four override cells + one resolve cell.
    - Each override applies dataclasses.replace on top of the automatic value.
    - An empty overrides dict means that dimension stays automatic.
    - EXIT_POLICY = None means each strategy keeps its assigned default.

Downstream the notebook references only the handles df, SYMBOL, INTERVAL, STRAT_CONFIG, EXIT_POLICY, TRADING_CONFIG. \
So "automatic vs manual" is decided in one place and the rest of the notebook is unchanged either way.

*Example for a new exit preset:* \
__Inject in the notebook:__ \
from engine.strategy_configurator import EXIT_PRESETS \
strat = OrderBlockStrategy(cfg, exit_policy=EXIT_PRESETS\["fixed_2pct_rr3"\]()) \
Backtester(strat, ...).run(df, interval=INTERVAL) \
__Sweep across the whole menu:__ \
for name in EXIT_PRESETS: \
    strat = SuperTrendStrategy(cfg, exit_policy=EXIT_PRESETS\[name\]()) \
    ...





## Fees: basis points (bps) vs percent (%)

Bybit quotes fees in **%**, but the engine stores and computes everything in **basis points (bps)**. \
These are the same quantity in different units — converting is just a unit swap, not a calculation:

| Unit | Value |
|------|-------|
| 1 bp | 0.01% |
| 100 bps | 1% |
| Bybit taker 0.04% | 4 bps |
| Bybit taker 0.055% | 5.5 bps |

**Conversion:** bps = percent × 100 \
So Bybit's 0.055% → multiply by 100 → store 5.5 bps.

### Why the engine uses bps

P&L is computed in bps, so costs share the same unit and subtract cleanly. In PositionState.exit():

raw_bps = (price - entry) / entry * 10_000   # fraction → bps \
trade.pnl_bps = raw_bps - cost_bps           # same unit, plain subtraction

The × 10_000 is the fraction → bps conversion (×100 to get %, ×100 again to get bps). \
Using bps avoids tiny decimals like 0.0004 and keeps fees, slippage, and returns on one consistent scale.

## Trade Direction 

- core.py class Direction: has no option 'BOTH'
- TradeDirection does: there is an option 'BOTH'

1. Direction (in core.py) is the **physical side of one concrete trade** — a single position can't be long and short at once. \
It's stamped on every Trade and Signal, and several pieces of code branch on it assuming exactly one of two values:
- **P&L in PositionState.exit()** — (price − entry) for a long, (entry − price) for a short. A BOTH trade has no defined P&L formula.
- **PositionState.update_peak()** — high-water (max) for longs, low-water (min) for shorts. BOTH has no defined trailing-stop direction.
- **state.enter(direction, …)** opens one position with one side; chart labels read "Long Entry" / "Short Entry".

Adding BOTH here would be a category error — it's not a third side, it's the *absence* of a single side, and every two-way branch above would need an undefined third case.

2. BOTH lives on **TradeDirection** (in trade_configurator.py) instead, because that's a different kind of thing: a **policy/permission over many trades** ("which sides is this run allowed to take"). \
"Both" is meaningful as a permission set; it's meaningless as the sign of one fill.

### The two are orthogonal

| | Direction (core.py) | TradeDirection (trade_configurator.py) |
|---|---|---|
| what | sign of **one** trade | allowed sides for the **run** |
| values | LONG / SHORT | LONG / SHORT / **BOTH** |
| set by | strategy, per signal (computed) | config, once (ACTIVE_TRADE / --direction) |
| BOTH meaningful? | no — a position has one side | yes — it's a filter over all signals |

So the gate (TradeDirection.BOTH) says "allow longs and shorts," while the strategy still emits each individual trade as a definite Direction.LONG or Direction.SHORT. \
Keeping Direction binary is what lets the P&L and trailing-stop math stay branch-clean.


## How does a startegy choose what direction to trade?

**The strategy's own signal logic decides the sign.**

A strategy chooses direction itself, per bar, in its on_bar() logic — there's no central direction picker. \
It evaluates its indicators and calls either state.enter(Direction.LONG, …) or state.enter(Direction.SHORT, …). \
The chosen side is hardcoded into each strategy's entry branches.

The inverse variant (_inv strategy) is a separate class that flips the same signal: enters the opposite side. \

So **choosing direction** happens at the strategy layer in two ways: 
- which signal logic fires the sign, and 
- which variant (base vs _inv) you run

The **TradeDirection gate** (TradingConfig.direction) doesn't choose — it can only veto. \
The strategy still emits a definite Direction.LONG/SHORT; if the run's gate disallows that side, state.enter() returns None and the attempt is counted as a suppressed entry.

So the flow is: \
strategy.on_bar()  →  picks Direction.LONG/SHORT  →  state.enter(direction, …) \
                                                         ├─ side allowed?  → opens the trade \
                                                         └─ side gated/halted? → None (suppressed)

**How to test how a strategy performs only in LONG**

Set the direction gate to LONG — the gate keeps only the strategy's long entries and suppresses every short.

3 ways:
1. In a notebook (recommended — inline, no global edit):
```python
from engine.backtester import Backtester
from engine.strategy_configurator import StrategyConfig
from engine.strategies import SuperTrendStrategy
from engine.trade_configurator import TradingConfig, TradeDirection
from engine.data_configurator import load_data, ACTIVE

df = load_data()
result = Backtester(
    SuperTrendStrategy(StrategyConfig()),
    symbol=ACTIVE.symbol,
    trading_config=TradingConfig(direction=TradeDirection.LONG),   # ← long-only
).run(df, interval=ACTIVE.interval)
print(result.summary())
```

2. Project-wide: \
Edit the ACTIVE_TRADE block in trade_configurator.py → direction = TradeDirection.LONG, then every notebook (and the CLI) inherits it.

3. CLI: \
python -m engine --strategy supertrend --direction long

Compare all three sides in one loop:
```python
for d in (TradeDirection.BOTH, TradeDirection.LONG, TradeDirection.SHORT):
    r = Backtester(SuperTrendStrategy(StrategyConfig()), symbol=ACTIVE.symbol,
                   trading_config=TradingConfig(direction=d)).run(df, interval=ACTIVE.interval)
    print(f"{d.value:5} | P&L {r.total_pnl_bps:+8.1f} bps | {r.total_trades} trades | {r.suppressed_entries} suppressed")
```

Two things to get right:
1. Use the base strategy, not the _inv variant. \
supertrend + direction=long = the strategy's long setups only — exactly what you want. \
supertrend_inv + direction=long tests something different (it disengages an inversion leg) and will print a warning.
2. Long-only is not just "the long trades from the both-direction run." \
Because the engine holds one position at a time, suppressing a short leaves the book flat, which frees it to take a later long the both-run was too busy (in a short) to take. \
So trade count and timing legitimately differ — that's the true long-only equity path, which is what you're after. \
Check result.suppressed_entries (now shown in summary()) to see how many shorts were dropped.

**Conclusion:**
- The strategy decides the side from its indicators
- The config can only allow or block that side, never change it
- If you want the opposite side of a setup: run the _inv variant — not a config flag
- If you want to test a strategy performance on long entries only:  set the direction gate to LONG (via trade_configurator.py)

## Exit Parameters Sweep

For sweeping: skip the presets and build the parameterized classes directly, injecting via exit_policy.

from engine.exits import ChandelierStop, CompositeExit, AtrStop, RrTarget

for mult in (2.0, 2.5, 3.0): \
    strat = SuperTrendStrategy(cfg, exit_policy=ChandelierStop(mult)) \
    Backtester(strat, symbol=SYMBOL, trading_config=ACTIVE_TRADE).run(df, interval=INTERVAL)

Or a 2-D SL×RR sweep: \
for a in (1.0, 1.5, 2.0): \
    for rr in (1.5, 2.0, 3.0): \
        strat = SuperTrendStrategy(cfg, exit_policy=CompositeExit(AtrStop(a), RrTarget(rr))) \
        ...

## Strategy testing



- **Backtest:**
    - Run the strategy once over a fixed stretch of history and look at the result.
    - Tells you: could this have made money at all? A quick check that an edge could exist.
    - Easy to fool yourself — you can tune the parameters until that one window looks great (overfitting).
    - Since it's easy to overfit, don't trust the "best" params it hands you.
- **Walk-forward (rolling):**
    - Repeat the test over many shifting windows: pick (or tune) on an earlier slice, then check on the next unseen slice, then slide forward and repeat.
    - Tells you: does it keep working on data it wasn't tuned on, across different market periods?
    - No separate validation backtest is needed afterward: walk-forward already is out-of-sample backtesting, \
      so its stitched-together out-of-sample result is your real verdict. \
      A later full-history backtest is only for eyeballing the equity curve or refitting final params for live.
- **Sweep:**
    - Belongs inside walk-forward, on the training slice only — never on the test slice (that's curve-fitting).
    - Sweep params on each training slice, score them on the next unseen slice, then slide forward and repeat. 

What each contributes:
- Backtest \
  → a quick sanity check + the raw edge. \
  First filter.
- Walk-forward/rolling \
  → robustness + honesty. \
  Shows if the edge keeps working out-of-sample, across different market periods, and is stable over time, not a one-window fluke.

In short:
- A backtest = it can work, walk-forward = it keeps working.
- Sweep to choose but walk-forward to trust the choice.
- Trust a strategy only after the rolling test, not the single backtest.
